### DS4420 Project: Indoor Scene Recognition
Colin Chu and Ethan Fang

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras.datasets import mnist
from keras.models import Model
from keras.layers import Dense, Input
from keras.layers import Conv2D, MaxPooling2D, Flatten
from keras.optimizers import SGD
from keras.losses import binary_crossentropy
from sklearn.model_selection import train_test_split
from PIL import Image, ImageOps
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def folder_to_numpy(folder_path, target_size=(128, 128)):
    images = []

    for i, subfolder in enumerate(os.listdir(folder_path)):
      print(f'Subfolder {i+1}/{len(os.listdir(folder_path))}')
      subfolder_path = os.path.join(folder_path, subfolder)

      for fname in os.listdir(subfolder_path):
        fpath = os.path.join(subfolder_path, fname)

        with Image.open(fpath) as img:
                img = img.convert('L')
                img = ImageOps.pad(img, target_size, color=0)
                images.append(np.array(img, dtype=np.uint8))

    return np.array(images, dtype=np.uint8)

In [ ]:
images_dir = '/content/drive/MyDrive/ds4420/DS4420 project/Images'
# comment this part out for Ethan
#images_dir = '/content/drive/MyDrive/project/Images'

scene_folders = os.listdir(images_dir)
all_data = []
all_labels = []

for label, folder_name in enumerate(scene_folders):
  print(f'Folder {label+1}/{len(scene_folders)}')
  folder_path = os.path.join(images_dir, folder_name)
  data = folder_to_numpy(folder_path)
  labels = np.full(data.shape[0], label)
  all_data.append(data)
  all_labels.append(labels)

indoor_data = np.concatenate(all_data, axis=0)
y = np.concatenate(all_labels, axis=0)

Folder 1/5
Subfolder 1/12
Subfolder 2/12
Subfolder 3/12
Subfolder 4/12
Subfolder 5/12
Subfolder 6/12
Subfolder 7/12
Subfolder 8/12
Subfolder 9/12
Subfolder 10/12
Subfolder 11/12
Subfolder 12/12
Folder 2/5
Subfolder 1/14
Subfolder 2/14
Subfolder 3/14
Subfolder 4/14
Subfolder 5/14
Subfolder 6/14
Subfolder 7/14
Subfolder 8/14
Subfolder 9/14
Subfolder 10/14
Subfolder 11/14
Subfolder 12/14
Subfolder 13/14
Subfolder 14/14
Folder 3/5
Subfolder 1/15
Subfolder 2/15
Subfolder 3/15
Subfolder 4/15
Subfolder 5/15
Subfolder 6/15
Subfolder 7/15
Subfolder 8/15
Subfolder 9/15
Subfolder 10/15
Subfolder 11/15
Subfolder 12/15
Subfolder 13/15
Subfolder 14/15
Subfolder 15/15
Folder 4/5
Subfolder 1/11
Subfolder 2/11
Subfolder 3/11
Subfolder 4/11
Subfolder 5/11
Subfolder 6/11
Subfolder 7/11
Subfolder 8/11
Subfolder 9/11
Subfolder 10/11
Subfolder 11/11
Folder 5/5
Subfolder 1/15
Subfolder 2/15
Subfolder 3/15
Subfolder 4/15
Subfolder 5/15
Subfolder 6/15
Subfolder 7/15
Subfolder 8/15
Subfolder 9/15
Subfolder 10/1

In [ ]:
# split the data into training and test
x_train, x_test, y_train, y_test = train_test_split(indoor_data, y, test_size=0.2, random_state=1, stratify=y)

# Normalize image data
x_train = x_train / 255.0
x_test = x_test / 255.0

# Reshape to add channel dimension for Keras
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

In [ ]:
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization

inpx = Input(shape=(128, 128, 1))

con_layer = Conv2D(32, kernel_size=(3,3), activation='relu', padding='same')(inpx)
con_layer = BatchNormalization()(con_layer)
pool_layer = MaxPooling2D(pool_size=(2,2))(con_layer)
drop_conv1 = Dropout(0.25)(pool_layer)

con_layer2 = Conv2D(64, kernel_size=(3,3), activation='relu', padding='same')(drop_conv1)
pool_layer2 = MaxPooling2D(pool_size=(2,2))(con_layer2)
drop_conv2 = Dropout(0.25)(pool_layer2)

con_layer3 = Conv2D(128, kernel_size=(3,3), activation='relu', padding='same')(drop_conv2)
pool_layer3 = MaxPooling2D(pool_size=(2,2))(con_layer3)
drop_conv3 = Dropout(0.25)(pool_layer3)

flat_G = Flatten()(drop_conv3)
hid_layer = Dense(256, activation='relu')(flat_G)
dropout = Dropout(0.5)(hid_layer)
hid_layer2 = Dense(128, activation='relu')(dropout)
dropout2 = Dropout(0.5)(hid_layer2)
out_layer = Dense(5, activation='softmax')(dropout2)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import sparse_categorical_crossentropy

model = Model([inpx], out_layer)
model.compile(optimizer=Adam(learning_rate=0.001), loss=sparse_categorical_crossentropy , metrics=['accuracy'])

In [ ]:
model.fit(x_train, y_train, batch_size=32, epochs=30)

Epoch 1/25
139/392 ━━━━━━━━━━━━━━━━━━━━ 7:20 2s/step - accuracy: 0.2520 - loss: 2.6661

KeyboardInterrupt: 

In [ ]:
score = model.evaluate(x_test, y_test, verbose=0)
print('loss=', score[0])
print('accuracy=', score[1])

In [ ]:
from tensorflow.keras.models import Model

feature_extractor = Model(inputs=model.input,
                          outputs=model.layers[-2].output)

train_features = feature_extractor.predict(x_train)
test_features  = feature_extractor.predict(x_test)

In [ ]:
train_df = pd.DataFrame(train_features)
train_df['label'] = y_train

test_df = pd.DataFrame(test_features)
test_df['label'] = y_test

train_df.to_csv('/content/drive/MyDrive/ds4420/DS4420 project/train_features.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/ds4420/DS4420 project/test_features.csv',  index=False)